# LangChain Tool Calling with OpenAI
Ví dụ định nghĩa, bind và gọi tools bằng LangChain.

In [1]:
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import (
    ArxivLoader,
    WebBaseLoader,
    WikipediaLoader,
)
import arxiv
import requests
import time
import wikipedia
import xml.etree.ElementTree as ET
from urllib.parse import quote
from langchain_core.documents import Document

load_dotenv()

C:\Users\QUAN\AppData\Local\Temp\ipykernel_28152\2570626968.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [3]:
# wikipedia 1.4.0 builds an HTTP API URL; Wikimedia requires HTTPS.
_original_set_lang = wikipedia.set_lang
def set_wikipedia_lang_https(prefix: str) -> None:
    _original_set_lang(prefix)
    wikipedia.wikipedia.API_URL = wikipedia.wikipedia.API_URL.replace("http://", "https://", 1)
    
wikipedia.set_lang = set_wikipedia_lang_https
wikipedia.set_user_agent("LangChainToolCallingNotebook/1.0 (educational use)")

# arxiv 1.x also defaults to HTTP, which is blocked by many environments.
arxiv.Client.query_url_format = arxiv.Client.query_url_format.replace("http://", "https://", 1)

## Define tools

In [4]:
@tool
def reverse_string(input_string: str) -> str:
    """Reverse a string"""
    output = input_string[::-1]
    return output

reversed_string = reverse_string.invoke({"input_string": "Hello Cybersoft"})
print(reversed_string)

tfosrebyC olleH


In [5]:
def format_documents(documents, max_chars: int = 4000) -> str:
    """Convert loaded documents into compact text for the model."""
    if not documents:
        return "Không tìm thấy tài liệu."
    sections = []
    for index, document in enumerate(documents, start=1):
        title = document.metadata.get("title", f"Tài liệu {index}")
        source = document.metadata.get("source", document.metadata.get("entry_id", "Không rõ nguồn"))
        sections.append(f"Tiêu đề: {title}\nNguồn: {source}\nNội dung:\n{document.page_content[:max_chars]}")
    return "\n\n---\n\n".join(sections)

def load_with_retry(loader_factory, attempts: int = 3, delay: float = 1.0):
    """Retry transient loader/network failures, then raise the last error."""
    last_error = None
    for attempt in range(attempts):
        try:
            documents = loader_factory().load()
            if documents:
                return documents
        except Exception as exc:
            last_error = exc
        if attempt < attempts - 1:
            time.sleep(delay * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []

@tool
def search_wikipedia(query: str, top_k: int = 2) -> str:
    """Tìm thông tin bách khoa trên Wikipedia."""
    try:
        # Call WikipediaLoader
        documents = load_with_retry(lambda: WikipediaLoader(query=query, load_max_docs=top_k, lang="vi"))
        return format_documents(documents)
    except Exception as loader_exc:
        # Fallback for wikipedia 1.4.0/API responses that are not valid JSON.
        try:
            # Avoid another API request (and possible 429): load the likely article URL directly.
            article_url = f"https://vi.wikipedia.org/wiki/{quote(query.strip().replace(' ', '_'))}"
            return format_documents(WebBaseLoader(web_paths=(article_url,), requests_kwargs={"timeout": 20}).load())
        except Exception as fallback_exc:
            return (f"Wikipedia hiện không truy cập được: primary={type(loader_exc).__name__}: {loader_exc}; "
                    f"fallback={type(fallback_exc).__name__}: {fallback_exc}")

@tool
def search_arxiv(query: str, top_k: int = 2) -> str:
    """Tìm các bài nghiên cứu khoa học trên arXiv."""
    try:
        # Call ArxivLoader
        documents = load_with_retry(lambda: ArxivLoader(query=query, load_max_docs=top_k, load_all_available_meta=True))
        return format_documents(documents)
    except Exception as loader_exc:
        # Fallback when arxiv 1.x/feedparser cannot expose the HTTP status.
        try:
            response = requests.get(
                "https://export.arxiv.org/api/query",
                params={"search_query": query, "start": 0, "max_results": top_k},
                headers={"User-Agent": "LangChainToolCallingNotebook/1.0 (educational use)"},
                timeout=30,
            )
            response.raise_for_status()
            root = ET.fromstring(response.content)
            ns = {"atom": "http://www.w3.org/2005/Atom"}
            documents = []
            for entry in root.findall("atom:entry", ns):
                title = " ".join((entry.findtext("atom:title", default="Untitled", namespaces=ns)).split())
                summary = " ".join((entry.findtext("atom:summary", default="", namespaces=ns)).split())
                source = entry.findtext("atom:id", default="", namespaces=ns)
                documents.append(Document(page_content=summary, metadata={"title": title, "source": source}))
            return format_documents(documents)
        except Exception as fallback_exc:
            return (f"arXiv hiện không truy cập được: primary={type(loader_exc).__name__}: {loader_exc}; "
                    f"fallback={type(fallback_exc).__name__}: {fallback_exc}")

@tool
def load_web_page(url: str) -> str:
    """Tải và đọc nội dung từ một URL cụ thể."""
    try:
        # Call WebBaseLoader
        return format_documents(WebBaseLoader(web_paths=(url,), requests_kwargs={"timeout": 15}).load())
    except Exception as exc:
        return f"Không tải được trang web: {type(exc).__name__}: {exc}"

# print(search_wikipedia.invoke({"query": "Trí tuệ nhân tạo", "top_k": 1})[:500])

In [6]:
# Multiply tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b."""
    return a * b

multiply.invoke({"a": 2, "b": 8})


16

## OpenAI model

In [7]:
import os

llm = ChatOpenAI(model="gpt-5-mini", api_key=os.getenv("OPENAI_API_KEY"))

## Binding and executing tools

In [8]:
tools = [reverse_string, multiply, search_wikipedia, search_arxiv, load_web_page]

model_with_tools = llm.bind_tools(tools)


In [18]:
response = model_with_tools.invoke("Tìm kiếm thông tin về bài báo khoa học 'Attention is all you need'")

for tool_call in response.tool_calls:
    print(tool_call)


{'name': 'search_arxiv', 'args': {'query': 'Attention is all you need', 'top_k': 2}, 'id': 'call_10GuH7IGXguaDMYYoSg7KSvv', 'type': 'tool_call'}


In [19]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 161, 'prompt_tokens': 266, 'total_tokens': 427, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E5or16uS3vJ5AMGy77SXfFXHbbdjH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f9d9e-624c-7f12-9bd9-fa159e919bdf-0', tool_calls=[{'name': 'search_arxiv', 'args': {'query': 'Attention is all you need', 'top_k': 2}, 'id': 'call_10GuH7IGXguaDMYYoSg7KSvv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 266, 'output_tokens': 161, 'total_tokens': 427, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_toke

## Additional tests
Các kiểm thử dưới đây xác nhận tool cơ bản, loader và OpenAI tool calling.

In [20]:
# Run every loader and display its full result
loader_cases = {
    "wikipedia": (search_wikipedia, {"query": "Trí tuệ nhân tạo", "top_k": 1}),
    "arxiv": (search_arxiv, {"query": "cat:cs.AI AND ti:agent", "top_k": 1}),
    "web": (load_web_page, {"url": "https://example.com"}),
}
loader_results = {}
for name, (loader_tool, arguments) in loader_cases.items():
    output = loader_tool.invoke(arguments)
    loader_results[name] = output
    print(f"\n{'=' * 20} {name.upper()} {'=' * 20}")
    print(output)

loader_results


==================== WIKIPEDIA ====================
Tiêu đề: Trí tuệ nhân tạo – Wikipedia tiếng Việt
Nguồn: https://vi.wikipedia.org/wiki/Tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o
Nội dung:




Trí tuệ nhân tạo – Wikipedia tiếng Việt




























Bước tới nội dung







Bảng chọn chính





Bảng chọn chính
chuyển sang thanh bên
ẩn



		Điều hướng
	


Trang ChínhNội dung chọn lọcBài viết ngẫu nhiênThay đổi gần đâyBáo lỗi nội dung





		Tương tác
	


Hướng dẫnGiới thiệu WikipediaCộng đồngThảo luận chungGiúp sử dụngLiên lạcTải lên tập tin



















Tìm kiếm











Tìm kiếm






















Giao diện
















Quyên góp

Tạo tài khoản

Đăng nhập








Công cụ cá nhân






Quyên góp


Tạo tài khoản


Đăng nhập





























Nội dung
chuyển sang thanh bên
ẩn




Đầu





1
Mục đích




Hiện/ẩn mục Mục đích





1.1
Suy luận và giải quyết vấn đề








1.2
Biểu diễn tri thức








1.3
Lập kế hoạch và ra quyết định








1.4
Học

{'wikipedia': "Tiêu đề: Trí tuệ nhân tạo – Wikipedia tiếng Việt\nNguồn: https://vi.wikipedia.org/wiki/Tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o\nNội dung:\n\n\n\n\nTrí tuệ nhân tạo – Wikipedia tiếng Việt\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nBước tới nội dung\n\n\n\n\n\n\n\nBảng chọn chính\n\n\n\n\n\nBảng chọn chính\nchuyển sang thanh bên\nẩn\n\n\n\n\t\tĐiều hướng\n\t\n\n\nTrang ChínhNội dung chọn lọcBài viết ngẫu nhiênThay đổi gần đâyBáo lỗi nội dung\n\n\n\n\n\n\t\tTương tác\n\t\n\n\nHướng dẫnGiới thiệu WikipediaCộng đồngThảo luận chungGiúp sử dụngLiên lạcTải lên tập tin\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nTìm kiếm\n\n\n\n\n\n\n\n\n\n\n\nTìm kiếm\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nGiao diện\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nQuyên góp\n\nTạo tài khoản\n\nĐăng nhập\n\n\n\n\n\n\n\n\nCông cụ cá nhân\n\n\n\n\n\n\nQuyên góp\n\n\nTạo tài khoản\n\n\nĐăng nhập\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNội dung\nchuyển sang thanh bên\nẩn\n\n\n\n\

In [21]:
# Automatic routing test with explicit source requests
routing_cases = [
    ("Chỉ dùng Wikipedia để tìm thông tin về Hà Nội.", "search_wikipedia"),
    ("Chỉ dùng arXiv để tìm nghiên cứu về large language model agents.", "search_arxiv"),
    ("Hãy đọc URL https://example.com bằng công cụ tải trang web.", "load_web_page"),
]
for prompt, expected_name in routing_cases:
    routed = model_with_tools.invoke(prompt)
    assert routed.tool_calls, f"No tool call for: {prompt}"
    assert routed.tool_calls[0]["name"] == expected_name, routed.tool_calls
    print(f"PASS: automatic routing -> {expected_name}")

PASS: automatic routing -> search_wikipedia
PASS: automatic routing -> search_arxiv
PASS: automatic routing -> load_web_page
